# Module `algorithms/benchmark.py`

Ce notebook presente le harnais de benchmark statique. Il permet de comparer les metaheuristiques sur un ensemble d'instances generees et de produire trois figures canoniques : qualite, ecart au meilleur, temps d'execution.

Le benchmark statique est distinct du benchmark dynamique (`dynamic_benchmark.py`) qui introduit un simulateur a chaque tour.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent / "src"))

from cesipath.algorithms import (
    genetic_algorithm,
    grasp,
    plot_benchmark_gap,
    plot_benchmark_quality,
    plot_benchmark_runtime,
    run_benchmark,
    simulated_annealing,
    summarize_benchmark,
    tabu_search,
)

## 1. Anatomie d'une ligne de benchmark

`run_benchmark` retourne une `list[dict]`. Chaque ligne correspond a un (algo x instance). Cles :

| Cle | Signification |
|---|---|
| `size` | Taille du graphe (valeur de `sizes`) |
| `seed` | Graine de l'instance (valeur de `seeds`) |
| `algo` | Nom donne dans le dict `algos` |
| `cost` | Cout total de la solution |
| `runtime` | Temps d'execution en secondes |
| `routes` | Nombre de tournees |

Pour un couple `(sizes, seeds)`, le harnais genere une seule instance par graine (via `GraphGenerator`), puis applique chaque algorithme sur cette meme instance. Cela rend la comparaison equitable : tous les algorithmes voient exactement le meme graphe.

## 2. Lancer un petit benchmark

On compare quatre algorithmes sur un jeu reduit pour rester rapide (quelques secondes). En pratique, on utilise plus de graines et de tailles.

In [ ]:
algos = {
    "grasp": grasp,
    "sa":    simulated_annealing,
    "tabu":  tabu_search,
    "ga":    genetic_algorithm,
}

algo_kwargs = {
    "grasp": {"max_iterations": 10, "rcl_alpha": 0.2},
    "sa":    {"max_iterations": 200, "initial_temperature": 100.0, "cooling_rate": 0.95},
    "tabu":  {"max_iterations": 60, "tabu_tenure": 7},
    "ga":    {"generations": 30, "population_size": 20, "memetic": True},
}

results = run_benchmark(
    sizes=[6, 8, 10],
    seeds=[1, 2, 3],
    algos=algos,
    algo_kwargs=algo_kwargs,
    verbose=False,
)
print(f"{len(results)} lignes produites")
print("Exemple :", results[0])

## 3. Resume agrege

`summarize_benchmark` regroupe par `(size, algo)` et renvoie moyenne / min / max du cout ainsi que le temps moyen. Pratique pour un tableau comparatif.

In [ ]:
summary = summarize_benchmark(results)
print(f"{'size':>4}  {'algo':<6}  {'n':>2}  {'cost_mean':>10}  {'cost_min':>10}  {'cost_max':>10}  {'time_mean':>10}")
for row in summary:
    print(f"{row['size']:>4}  {row['algo']:<6}  {row['n_instances']:>2}  "
          f"{row['cost_mean']:>10.2f}  {row['cost_min']:>10.2f}  {row['cost_max']:>10.2f}  "
          f"{row['runtime_mean']:>10.3f}")

## 4. Figure qualite : boxplot du cout

`plot_benchmark_quality(results)` trace un boxplot du cout par taille, une couleur par algorithme. Chaque boite resume la distribution des couts sur les differentes graines.

Lecture : plus la mediane est basse, plus l'algorithme est bon ; plus la boite est etroite, plus il est stable.

In [ ]:
fig = plot_benchmark_quality(results)
fig

## 5. Figure gap : ecart relatif au meilleur

`plot_benchmark_gap(results)` calcule, pour chaque instance, le meilleur cout obtenu (quel que soit l'algo), puis l'ecart relatif moyen par algo :

```text
gap_i = 100 * (cost_algo - cost_best) / cost_best
```

Un algo optimal sur toutes les instances aurait un gap moyen de 0. Cette figure est particulierement utile pour comparer sur des tailles differentes : elle normalise la grandeur des couts.

In [ ]:
fig = plot_benchmark_gap(results)
fig

## 6. Figure runtime : temps d'execution

`plot_benchmark_runtime(results, log_scale=True)` trace le temps moyen en fonction de la taille, une ligne par algo. Echelle logarithmique par defaut (les metaheuristiques sont generalement polynomiales, mais avec des constantes tres differentes).

In [ ]:
fig = plot_benchmark_runtime(results, log_scale=False)
fig

## 7. Sauvegarder les trois figures

`save_benchmark_figures(results, directory=None)` :

- cree ou reutilise le dossier cible (`DEFAULT_IMAGE_DIR = algorithms/image/` par defaut) ;
- trouve le plus petit `N >= 1` tel qu'aucun des fichiers `benchmark_{quality,gap,runtime}_N.png` n'existe deja (compteur propre a ce prefixe, independant de `save_solution_plot` et `save_dynamic_benchmark_figures`) ;
- ecrit `benchmark_quality_N.png`, `benchmark_gap_N.png`, `benchmark_runtime_N.png` ;
- renvoie un `dict` avec les trois chemins.

Note : `algorithms/image/` est dans `.gitignore`. On n'y commit **jamais** de PNG.

## 8. Conseils pour un benchmark serieux

- **Plus de graines**. Un benchmark a 10 graines par taille est le minimum raisonnable pour lisser le bruit.
- **Variez les tailles**. 6-8 tailles distribuees (par exemple 10, 20, 30, 50, 80, 120) montrent l'evolution des algos.
- **Memes budgets**. Donnez un budget d'effort comparable aux algos via `algo_kwargs` (nombre d'iterations, population, temperature initiale).
- **Reproductibilite**. Le parametre `seed` est passe automatiquement a chaque algo via `kwargs.setdefault`, ce qui rend chaque ligne reproductible.
- **Interpretation**. Pour conclure qu'un algo `A` bat un algo `B`, verifiez que le gap moyen est nettement en faveur de `A` ET que le temps est du meme ordre de grandeur.

## 9. Benchmark dynamique

Pour comparer des strategies sur scenarios dynamiques (plan fige vs re-optimisation a chaque retour depot), utiliser `run_dynamic_benchmark` et les `plot_dynamic_*` associes. Ils sont documentes dans le code de `algorithms/dynamic_benchmark.py` et exposes au niveau du package (`cesipath.run_dynamic_benchmark`).

## 10. Choix de l'algorithme retenu : Tabou

### Critere de decision

On retient un unique algorithme a deployer. Le critere principal est le **rang combine qualite + vitesse** sur l'ensemble des instances testees (550 runs statiques, 1 551 runs dynamiques, benchmark `afternoon / balanced`). Un bon rang combine signifie qu'un algorithme n'est jamais le pire ni en qualite ni en temps — c'est le bon critere pour un usage en production ou les deux dimensions comptent simultanement.

### Synthese statique (550 runs)

| Algorithme | Ecart moyen | Dans 1 % du meilleur | Temps median | Rang combine |
|---|---|---|---|---|
| GA         | 0.14 %      | 95.5 %               | 16.0 s       | 2.58         |
| GRASP      | 0.70 %      | 75.8 %               | 0.44 s       | 2.31         |
| **Tabou**  | **1.25 %**  | **59.6 %**           | **0.069 s**  | **1.94**     |
| SA         | 2.34 %      | 42.5 %               | 0.045 s      | 2.31         |

### Synthese dynamique, strategie `fixed` (1 551 runs)

| Algorithme | Ecart moyen | Dans 1 % du meilleur | Temps median | Rang combine |
|---|---|---|---|---|
| **Tabou**  | **5.64 %**  | **30.8 %**           | **0.33 s**   | **2.04**     |
| SA         | 6.43 %      | 25.1 %               | 0.30 s       | 2.17         |
| GRASP      | 5.04 %      | 37.1 %               | 0.65 s       | 2.62         |
| GA         | 4.10 %      | 44.6 %               | 14.4 s       | 3.11         |

### Justification

**GA** produit la meilleure qualite absolue en statique (0.14 % d'ecart), mais son temps median de 16 s le rend incompatible avec des re-optimisations dynamiques : chaque retour depot declencherait une resolution longue. Son rang combine en dynamique est le pire des quatre (3.11).

**GRASP** offre un bon equilibre en statique (rang 2.31), mais sa construction RCL est couteuse et ne tire pas de benefice net de la strategie `reopt` en dynamique (rang 2.62).

**SA** est le plus rapide en temps brut, mais la qualite de ses solutions est la plus faible dans les deux modes.

**Tabou** est #1 en rang combine dans les deux modes (1.94 statique, 2.04 dynamique). Son ecart de 1.25 % reste proche du meilleur a un budget temps tres faible (0.069 s). Surtout, c'est le **seul algorithme pour lequel la strategie `reopt` est systematiquement benefique** : le gain moyen de **+0.525 %** sur le cout realise justifie le cout des re-optimisations, alors que GRASP, SA et GA voient ce gain neutre ou negatif. Le balayage exhaustif du voisinage converge rapidement depuis un etat degrade, la memoire tabou empechant les cycles qui piegeraient une descente gloutonne.

**Conclusion** : Tabou est retenu. Il offre le meilleur rapport qualite/vitesse sur les deux scenarios, et est le seul algorithme a tirer pleinement parti de la re-optimisation dynamique.